In [1]:
!pip install beautifulsoup4

In [2]:
import requests

resp = requests.get(
    "https://www.boxofficemojo.com/title/tt0468569/",  # The Dark Knight, known to have full data
    headers={"User-Agent": "Mozilla/5.0"}
)
print(resp.status_code)  # should be 200 — if not, that's the whole problem right there

with open("bom_sample.html", "w", encoding="utf-8") as f:
    f.write(resp.text)

200


In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

df = pd.read_csv("tmdb_movies.csv")

def scrape_boxofficemojo(imdb_id):
    url = f"https://www.boxofficemojo.com/title/{imdb_id}/"
    resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    soup = BeautifulSoup(resp.text, "html.parser")

    data = {"imdb_id": imdb_id, "mpaa_rating": None, "distributor": None}

    for label_tag in soup.find_all(string=True):
        text = label_tag.strip()
        if not text:
            continue
        if text == "MPAA":
            value = label_tag.find_parent().find_next_sibling()
            if value:
                data["mpaa_rating"] = value.get_text(strip=True)
        elif "Distributor" in text:
            value = label_tag.find_parent().find_next_sibling()
            if value:
                data["distributor"] = value.get_text(strip=True)

    return data

# test on ONE known-good movie first — don't loop over your whole list yet
print(scrape_boxofficemojo("tt0468569"))

{'imdb_id': 'tt0468569', 'mpaa_rating': 'PG-13', 'distributor': 'Warner Bros.See full company information'}


In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from tqdm import tqdm

df = pd.read_csv("tmdb_movies.csv")

def scrape_boxofficemojo(imdb_id):
    url = f"https://www.boxofficemojo.com/title/{imdb_id}/"
    resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=15)
    soup = BeautifulSoup(resp.text, "html.parser")

    data = {"imdb_id": imdb_id, "mpaa_rating": None, "distributor": None}

    for label_tag in soup.find_all(string=True):
        text = label_tag.strip()
        if not text:
            continue
        if text == "MPAA":
            value = label_tag.find_parent().find_next_sibling()
            if value:
                data["mpaa_rating"] = value.get_text(strip=True)
        elif "Distributor" in text:
            value = label_tag.find_parent().find_next_sibling()
            if value:
                clean = value.get_text(strip=True).split("See full")[0].strip()
                data["distributor"] = clean

    return data

# quick sanity check before the full run
print(scrape_boxofficemojo("tt0468569"))

{'imdb_id': 'tt0468569', 'mpaa_rating': 'PG-13', 'distributor': 'Warner Bros.'}


In [5]:
import os

OUT_PATH = "boxofficemojo_predictors.csv"

# resume: skip ids already saved by a previous (interrupted) run
done = set()
if os.path.exists(OUT_PATH):
    done = set(pd.read_csv(OUT_PATH)["imdb_id"])

todo = [i for i in df["imdb_id"].dropna() if i not in done]
print(f"{len(done)} already done, {len(todo)} to go")

def scrape_with_retry(imdb_id, attempts=3):
    for attempt in range(attempts):
        try:
            return scrape_boxofficemojo(imdb_id)
        except requests.exceptions.RequestException as e:
            print(f"{imdb_id}: attempt {attempt + 1} failed ({type(e).__name__})")
            time.sleep(10 * (attempt + 1))  # wait longer each time
    return None  # give up on this one, keep going

results = []
for n, imdb_id in enumerate(tqdm(todo), 1):
    row = scrape_with_retry(imdb_id)
    if row is not None:
        results.append(row)
    time.sleep(1)

    # checkpoint every 50 movies so a crash doesn't lose everything
    if n % 50 == 0 or n == len(todo):
        pd.DataFrame(results).to_csv(
            OUT_PATH, mode="a", index=False, header=not os.path.exists(OUT_PATH)
        )
        results = []

1938 already done, 0 to go


0it [00:00, ?it/s]
